# SQL Analysis — Margin Ranking by Region

Uses SQLite (loaded from the cleaned CSV) to answer: within each region, which sub-categories have the best and worst profit margins?

This uses a window function (`RANK() OVER (PARTITION BY ... ORDER BY ...)`) rather than a plain GROUP BY, since we want a rank *within* each region, not just an overall ranking.

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/Sample - Superstore.csv", encoding="latin1")
df = df.drop_duplicates(subset=[c for c in df.columns if c != "Row ID"], keep="first")

conn = sqlite3.connect(":memory:")
df.to_sql("orders", conn, index=False, if_exists="replace")

pd.read_sql("SELECT COUNT(*) as row_count FROM orders", conn)

,row_count
0,9993


## Margin ranking by region (window function)

For each region, rank sub-categories by profit margin (profit ÷ sales). Using `RANK() OVER (PARTITION BY Region ORDER BY margin)` gives a rank *within* each region — this is the SQL equivalent of a `groupby` + rank in pandas, and demonstrates window-function usage explicitly rather than just aggregation.

In [2]:
query = """
WITH subcat_region AS (
    SELECT
        Region,
        "Sub-Category" AS sub_category,
        SUM(Sales) AS total_sales,
        SUM(Profit) AS total_profit,
        SUM(Profit) * 1.0 / SUM(Sales) AS margin
    FROM orders
    GROUP BY Region, "Sub-Category"
)
SELECT
    Region,
    sub_category,
    ROUND(total_sales, 2) AS total_sales,
    ROUND(total_profit, 2) AS total_profit,
    ROUND(margin, 4) AS margin,
    RANK() OVER (PARTITION BY Region ORDER BY margin ASC) AS worst_margin_rank
FROM subcat_region
ORDER BY Region, worst_margin_rank
"""

result = pd.read_sql(query, conn)
result

,Region,sub_category,total_sales,total_profit,margin,worst_margin_rank
0,Central,Furnishings,15254.37,-3906.22,-0.2561,1
1,Central,Appliances,23582.03,-2638.62,-0.1119,2
2,Central,Tables,39154.97,-3559.65,-0.0909,3
3,Central,Bookcases,24157.18,-1997.90,-0.0827,4
4,Central,Supplies,9467.37,-661.89,-0.0699,5
...,...,...,...,...,...,...
63,West,Fasteners,923.22,275.19,0.2981,13
64,West,Copiers,49749.24,19327.24,0.3885,14
65,West,Labels,5078.73,2303.12,0.4535,15
66,West,Paper,26663.72,12119.24,0.4545,16


In [3]:
result[result["worst_margin_rank"] <= 3]

,Region,sub_category,total_sales,total_profit,margin,worst_margin_rank
0,Central,Furnishings,15254.37,-3906.22,-0.2561,1
1,Central,Appliances,23582.03,-2638.62,-0.1119,2
2,Central,Tables,39154.97,-3559.65,-0.0909,3
17,East,Tables,39139.81,-11025.38,-0.2817,1
18,East,Supplies,10760.12,-1155.14,-0.1074,2
19,East,Bookcases,43819.33,-1167.63,-0.0266,3
34,South,Tables,43916.19,-4623.06,-0.1053,1
35,South,Machines,53890.96,-1438.89,-0.0267,2
36,South,Supplies,8318.93,1.88,0.0002,3
51,West,Bookcases,36004.12,-1646.51,-0.0457,1


### Findings

The worst-margin sub-category is **not the same everywhere**, but there's a clear pattern:

| Region | Worst sub-category | Margin |
|---|---|---|
| Central | Furnishings | -25.6% |
| East | Tables | -28.2% |
| South | Tables | -10.5% |
| West | Bookcases | -4.6% |

**Tables loses money in 3 of 4 regions** (Central, East, South) but is actually *profitable* in West (+1.75% margin, $1,483 profit on $84,755 sales). This is a genuinely useful, non-obvious finding: a blanket "stop discounting Tables" policy would be wrong for the West region specifically — worth investigating what's different about West's Tables pricing/discounting before applying a company-wide fix.

Supplies is inconsistent too — a strong loss-maker in East (-10.7%) but nearly break-even in South (+0.02%).